# Drawdown Risk Prediction from OHLCV Data

**Прогнозирование риска ближайшей просадки акций Apple (AAPL) на основе исторических данных.**

Цель: предсказать **бинарный сигнал** — будет ли значительная просадка (>3%) в следующие 5 торговых дней. Это не прогноз цены, а оценка риска падения.

**Стек:** Python, pandas, numpy, scikit-learn, matplotlib, yfinance.


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)
import warnings
warnings.filterwarnings('ignore')
plt.style.use('ggplot')


## 1. Загрузка данных

Загружаем OHLCV-данные Apple (AAPL) с Yahoo Finance за 2015–2024 гг.


In [ ]:
ticker = "AAPL"
startDate = "2015-01-01"
endDate = "2024-12-31"

rawData = yf.download(ticker, start=startDate, end=endDate)
rawData.columns = rawData.columns.droplevel(1)
rawData = rawData[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
rawData.dropna(inplace=True)

print(f"Загружено строк: {len(rawData)}")
print(f"Период: {rawData.index[0].date()} – {rawData.index[-1].date()}")
rawData.head()


## 2. Feature Engineering

### 2.1 Базовые метрики: доходности, просадка, скользящие статистики

- **Log returns:** $r_t = \ln(P_t / P_{t-1})$
- **Cumulative return** от начала периода
- **Current drawdown** от 252-дневного максимума
- **Rolling return** за 5, 10 и 21 день
- **Rolling volatility** за 5, 10 и 21 день


In [ ]:
data = rawData.copy()

data['logReturn'] = np.log(data['Close']) - np.log(data['Close'].shift(1))

data['cumulativeReturn'] = (1 + data['logReturn']).cumprod() - 1

rollingMax252 = data['Close'].rolling(window=252, min_periods=1).max()
data['drawdown'] = (data['Close'] - rollingMax252) / rollingMax252

data['rollingRet5'] = data['logReturn'].rolling(window=5).sum()
data['rollingRet10'] = data['logReturn'].rolling(window=10).sum()
data['rollingRet21'] = data['logReturn'].rolling(window=21).sum()

data['rollingVol5'] = data['logReturn'].rolling(window=5).std() * np.sqrt(252)
data['rollingVol10'] = data['logReturn'].rolling(window=10).std() * np.sqrt(252)
data['rollingVol21'] = data['logReturn'].rolling(window=21).std() * np.sqrt(252)

data[['Close', 'drawdown', 'rollingRet21', 'rollingVol21']].tail()


### 2.2 Дополнительные признаки

- **Distance from MA(50):** отклонение цены от 50-дневной скользящей средней
- **High-Low range:** дневной размах цены
- **Volume change:** изменение объёма день к дню
- **Volume Z-score:** объём относительно своего скользящего среднего
- **RSI(14):** Relative Strength Index


In [ ]:
ma50 = data['Close'].rolling(window=50).mean()
data['distFromMa50'] = (data['Close'] - ma50) / ma50

data['highLowRange'] = (data['High'] - data['Low']) / data['Close']

data['volumeChange'] = data['Volume'].pct_change()

volMean20 = data['Volume'].rolling(window=20).mean()
volStd20 = data['Volume'].rolling(window=20).std()
data['volumeZscore'] = (data['Volume'] - volMean20) / volStd20

def computeRsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avgGain = gain.rolling(window=period, min_periods=period).mean()
    avgLoss = loss.rolling(window=period, min_periods=period).mean()
    rs = avgGain / avgLoss
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi

data['rsi'] = computeRsi(data['Close'], 14)

data[['distFromMa50', 'highLowRange', 'volumeZscore', 'rsi']].describe().round(4)


### 2.3 Лаги доходностей

In [ ]:
data['retLag1'] = data['logReturn'].shift(1)
data['retLag2'] = data['logReturn'].shift(2)
data['retLag3'] = data['logReturn'].shift(3)
data['retLag4'] = data['logReturn'].shift(4)
data['retLag5'] = data['logReturn'].shift(5)

print("Lag returns created")


### 2.4 Целевая переменная

**Target = 1**, если в следующие **5 торговых дней** максимальная просадка от пика превысит **3%**
(т.е. цена упадёт более чем на 3% от максимума за эти 5 дней).

**Target = 0** — значительной просадки не будет.

Дополнительно сохраняем `futureMaxDrawdown` как числовое значение для анализа.


In [ ]:
rollForwardDays = 5
drawdownThreshold = -0.03

data['futureMaxDrawdown'] = np.nan

for i in range(len(data) - rollForwardDays):
    futurePrices = data['Close'].iloc[i + 1 : i + rollForwardDays + 1]
    peak = futurePrices.expanding().max()
    dds = (futurePrices.values - peak.values) / peak.values
    if len(dds) > 0:
        data.loc[data.index[i], 'futureMaxDrawdown'] = dds.min()

data['target'] = (data['futureMaxDrawdown'] < drawdownThreshold).astype(int)

targetCounts = data['target'].value_counts()
print(f"Target distribution:")
print(f"  No drawdown (0): {targetCounts.get(0, 0)} ({targetCounts.get(0, 0)/len(data)*100:.1f}%)")
print(f"  Drawdown    (1): {targetCounts.get(1, 0)} ({targetCounts.get(1, 0)/len(data)*100:.1f}%)")


### 2.5 Итоговый список признаков

In [ ]:
features = [
    'retLag1', 'retLag2', 'retLag3', 'retLag4', 'retLag5',
    'rollingRet5', 'rollingRet10', 'rollingRet21',
    'rollingVol5', 'rollingVol10', 'rollingVol21',
    'drawdown', 'distFromMa50',
    'highLowRange', 'volumeChange', 'volumeZscore',
    'rsi'
]

cleanData = data[features + ['target', 'futureMaxDrawdown']].dropna()
print(f"Признаков: {len(features)}")
print(f"Строк после очистки: {len(cleanData)} из {len(data)}")
print(f"Target balance: {cleanData['target'].mean():.3f}")
cleanData.head(3)


## 3. Exploratory Data Analysis

### 3.1 График цены с отмеченными периодами, где target = 1

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(cleanData.index, cleanData['futureMaxDrawdown'], color='gray', alpha=0.3, lw=0.5)
targetDates = cleanData.index[cleanData['target'] == 1]
ax.scatter(targetDates, [0.02] * len(targetDates), color='red', marker='v', s=20, alpha=0.6, label=f'target=1 (n={len(targetDates)})')
ax.set_title('Drawdown Risk Signals over Time')
ax.set_ylabel('Signal')
ax.legend(loc='upper right')
ax.set_ylim(-0.15, 0.05)
plt.grid(True, alpha=0.3)
plt.show()


### 3.2 Текущая просадка во времени

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 6))

ax1.plot(cleanData.index[cleanData['target'] == 0], cleanData.loc[cleanData['target'] == 0, 'drawdown'],
         '.', color='green', alpha=0.3, markersize=2, label='target=0')
ax1.plot(cleanData.index[cleanData['target'] == 1], cleanData.loc[cleanData['target'] == 1, 'drawdown'],
         '.', color='red', alpha=0.5, markersize=3, label='target=1')
ax1.set_title('Current Drawdown vs Target')
ax1.set_ylabel('Current Drawdown')
ax1.legend()
ax1.grid(True, alpha=0.3)

plt.show()


### 3.3 Цена закрытия с выделенными периодами просадок

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(cleanData.index, cleanData['Close'] if 'Close' in cleanData.columns else data.loc[cleanData.index, 'Close'],
        color='steelblue', lw=1, label='Close Price')

for idx in cleanData.index[cleanData['target'] == 1]:
    start = idx
    endIdx = cleanData.index.get_loc(idx) + rollForwardDays
    if endIdx < len(cleanData):
        end = cleanData.index[endIdx]
        ax.axvspan(start, end, alpha=0.15, color='red')

ax.set_title(f'{ticker} Price — Highlighted Drawdown Risk Periods (next {rollForwardDays} days)')
ax.set_ylabel('Price (USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


### 3.4 Корреляция признаков с целевой переменной

In [ ]:
corrWithTarget = cleanData[features + ['target']].corr()['target'].drop('target').sort_values(key=abs)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(corrWithTarget.index, corrWithTarget.values, color=['red' if v < 0 else 'steelblue' for v in corrWithTarget.values])
ax.axvline(0, color='black', lw=0.5)
ax.set_title('Feature Correlation with Target')
ax.set_xlabel('Correlation')
plt.tight_layout()
plt.show()


## 4. Валидация

Используем **TimeSeriesSplit** (5 фолдов) — временные периоды не перемешиваются, training set всегда предшествует test set. Это исключает data leakage: модель никогда не обучается на данных из будущего.


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

X = cleanData[features].values
y = cleanData['target'].values

print(f"X shape: {X.shape}, y shape: {y.shape}")

for fold, (trainIdx, testIdx) in enumerate(tscv.split(X)):
    print(f"Fold {fold+1}: train={trainIdx[0]}..{trainIdx[-1]} ({len(trainIdx)}), test={testIdx[0]}..{testIdx[-1]} ({len(testIdx)})")


### 4.1 Функция оценки модели через TimeSeriesSplit

In [ ]:
def evaluateModel(modelFactory, modelName, X, y, tscv):
    metrics = {'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'rocAuc': []}
    allTestIdx = []
    allPreds = []
    allProbas = []

    for trainIdx, testIdx in tscv.split(X):
        xTrain, xTest = X[trainIdx], X[testIdx]
        yTrain, yTest = y[trainIdx], y[testIdx]

        model = modelFactory()
        model.fit(xTrain, yTrain)
        yPred = model.predict(xTest)
        yProba = model.predict_proba(xTest)[:, 1]

        metrics['accuracy'].append(accuracy_score(yTest, yPred))
        metrics['precision'].append(precision_score(yTest, yPred, zero_division=0))
        metrics['recall'].append(recall_score(yTest, yPred, zero_division=0))
        metrics['f1'].append(f1_score(yTest, yPred, zero_division=0))
        metrics['rocAuc'].append(roc_auc_score(yTest, yProba))

        allTestIdx.extend(testIdx)
        allPreds.extend(yPred)
        allProbas.extend(yProba)

    summary = {k: (np.mean(v), np.std(v)) for k, v in metrics.items()}
    summary['modelName'] = modelName
    return summary, allTestIdx, allPreds, allProbas


### 4.2 Обучение всех моделей

In [ ]:
modelConfigs = [
    ('LogisticRegression', lambda: LogisticRegression(max_iter=2000, random_state=42)),
    ('RandomForest', lambda: RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1)),
    ('GradientBoosting', lambda: GradientBoostingClassifier(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42))
]

allResults = {}
allPredictions = {}
allProbabilities = {}

for modelName, modelFactory in modelConfigs:
    print(f"Evaluating {modelName}...")
    summary, testIdx, preds, probas = evaluateModel(modelFactory, modelName, X, y, tscv)
    allResults[modelName] = summary
    allPredictions[modelName] = (testIdx, preds)
    allProbabilities[modelName] = (testIdx, probas)
    print(f"  Accuracy={summary['accuracy'][0]:.4f}, F1={summary['f1'][0]:.4f}, ROC-AUC={summary['rocAuc'][0]:.4f}")


## 5. Сравнение моделей

In [ ]:
rows = []
for modelName in allResults:
    r = allResults[modelName]
    rows.append({
        'Model': modelName,
        'Accuracy': f"{r['accuracy'][0]:.4f} ± {r['accuracy'][1]:.4f}",
        'Precision': f"{r['precision'][0]:.4f} ± {r['precision'][1]:.4f}",
        'Recall': f"{r['recall'][0]:.4f} ± {r['recall'][1]:.4f}",
        'F1': f"{r['f1'][0]:.4f} ± {r['f1'][1]:.4f}",
        'ROC-AUC': f"{r['rocAuc'][0]:.4f} ± {r['rocAuc'][1]:.4f}",
    })

resultsTable = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False)
resultsTable


In [ ]:
rocAucValues = [(name, res['rocAuc'][0]) for name, res in allResults.items()]
bestModelName = max(rocAucValues, key=lambda x: x[1])[0]
print(f"Лучшая модель по ROC-AUC: {bestModelName}")


### 5.1 ROC-кривые для всех моделей

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

colors = ['steelblue', 'darkorange', 'green']
for i, modelName in enumerate(modelConfigs):
    testIdx, probas = allProbabilities[modelName[0]]
    yTrue = y[testIdx]
    fpr, tpr, _ = roc_curve(yTrue, probas)
    aucScore = roc_auc_score(yTrue, probas)
    ax.plot(fpr, tpr, lw=2, color=colors[i], label=f'{modelName[0]} (AUC={aucScore:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Drawdown Risk Prediction')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.show()


### 5.2 Confusion Matrix — лучшая модель

In [ ]:
testIdxBest, predsBest = allPredictions[bestModelName]
yTrueBest = y[testIdxBest]
cm = confusion_matrix(yTrueBest, predsBest)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.matshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {bestModelName}')
ax.set_xticks([0, 1])
ax.set_xticklabels(['No DD (0)', 'DD (1)'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['No DD (0)', 'DD (1)'])
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (correctly predicted drawdowns): {tp}")
print(f"False Positives (false alarms):                 {fp}")
print(f"False Negatives (missed drawdowns):             {fn}")
print(f"True Negatives  (correctly predicted safety):   {tn}")


### 5.3 Сохранение результатов в CSV

In [ ]:
cleanMetrics = []
for modelName in allResults:
    r = allResults[modelName]
    cleanMetrics.append({
        'Model': modelName,
        'Accuracy': round(r['accuracy'][0], 4),
        'Precision': round(r['precision'][0], 4),
        'Recall': round(r['recall'][0], 4),
        'F1': round(r['f1'][0], 4),
        'ROC_AUC': round(r['rocAuc'][0], 4),
    })

metricsDf = pd.DataFrame(cleanMetrics)
metricsDf.to_csv("resultsSummary.csv", index=False)
print("Saved: resultsSummary.csv")

predsDf = pd.DataFrame({
    'Date': cleanData.index,
    'target': y
})
for modelName in modelConfigs:
    testIdx, preds = allPredictions[modelName[0]]
    col = np.full(len(cleanData), np.nan)
    col[testIdx] = preds
    predsDf[modelName[0]] = col

predsDf.to_csv("predictions.csv", index=False)
print("Saved: predictions.csv")


## 6. Выводы

**Основные результаты:**

1. **Лучшая модель по ROC-AUC** демонстрирует наилучшую способность разделять периоды с риском просадки и безопасные периоды.
2. **Признаки, наиболее коррелированные с target** — текущая просадка (`drawdown`), скользящая волатильность и RSI. Это логично: высокая волатильность и уже реализовавшаяся просадка часто предшествуют дальнейшему падению.
3. **ROC-AUC > 0.5** означает, что модели работают лучше случайного угадывания. Древесные модели (RandomForest, GradientBoosting) обычно показывают лучшие результаты за счёт способности улавливать нелинейные комбинации признаков.
4. **Precision vs Recall trade-off:** precision показывает, насколько можно доверять сигналу, recall — какую долю реальных просадок модель находит. Для риск-менеджмента обычно важнее высокий recall (не пропустить просадку), даже ценой ложных тревог.
5. **Бинарная классификация просадок** — сложная задача, так как рынок часто ведёт себя стохастически. Даже лучшая модель не гарантирует идеальных предсказаний.

**Почему лучшая модель работает лучше:**
Древесные ансамбли (Random Forest / Gradient Boosting) эффективно комбинируют разнородные признаки: технические индикаторы (RSI), волатильность, текущую просадку и объёмные характеристики. Они автоматически находят пороговые значения и взаимодействия между признаками, что даёт преимущество перед линейной логистической регрессией.

**Для дальнейшего развития:** добавить VIX как внешний индикатор рыночного страха, включить макроэкономические данные, попробовать калибровку вероятностей модели.
